In [ ]:
%matplotlib inline
import warnings
# silence the harmless tqdm/ipywidgets notebook warning (cosmetic only)
warnings.filterwarnings("ignore", message="IProgress not found.*")

import numpy as np
import matplotlib.pyplot as plt

from iss_preprocess.io import load_tile_by_coors, get_roi_dimensions

# Relative path to the processed dataset: <project>/<mouse>/<chamber>
data_path = "toksozi_in-vivo-BRISC/BRAC12106.1c/chamber_01"

In [ ]:
# --- choose what to load -------------------------------------------------
prefix = "barcode_round_1_1"  # round/acquisition folder, e.g. "genes_round_1_1", "mcherry_neun_1_1"
roi = 27                       # ROI id
tile_x = 3                     # tile column  (X / first Pos index)
tile_y = 7                     # tile row     (Y / second Pos index)
suffix = "max"                # projection: "max", "median", "mean" or "fstack"
channel = 3                    # channel to plot (0-based camera index; far-red NeuN = 3)
# -------------------------------------------------------------------------

stack = load_tile_by_coors(
    data_path,
    tile_coors=(roi, tile_x, tile_y),
    suffix=suffix,
    prefix=prefix,
)
print(f"loaded {prefix} tile (roi={roi}, x={tile_x}, y={tile_y}) "
      f"-> shape {stack.shape} (X, Y, channels), dtype {stack.dtype}")

In [ ]:
# plot the chosen channel
img = stack[:, :, channel]

# robust contrast: signal is sparse on a dark background, so clip to percentiles
vmin, vmax = np.percentile(img, [1, 99.9])

plt.figure(figsize=(8, 8))
plt.imshow(img, cmap="gray", vmin=vmin, vmax=vmax)
plt.title(f"{prefix}  roi={roi}  tile=({tile_x},{tile_y})  ch={channel}  [{suffix}]")
plt.colorbar(fraction=0.046, pad=0.04)
plt.axis("off")
plt.show()

In [ ]:
# optional: tile-grid bounding box per ROI -> rows are (roi_id, max_x, max_y)
# valid tile indices for an ROI run 0..max_x and 0..max_y
get_roi_dimensions(data_path, prefix)

## Reference-tile registration demo

Walk through the step the pipeline runs first (`run_register_reference_tile`):
load one tile across **all rounds** and estimate the channel/round transforms.
If a round has no structure the shifts come back as `NaN` and the real pipeline
raises `"Reference tforms contain NaNs"`.

Below we run this **non-destructively** (nothing written to disk) for two
candidate `ref_tile`s, `[15, 3, 5]` and `[15, 3, 2]`, to compare them.

In [ ]:
# registration imports + parameters (read straight from this dataset's ops)
from iss_preprocess.io import load_ops, load_sequencing_rounds
from iss_preprocess.reg.rounds_and_channels import register_channels_and_rounds

ops = load_ops(data_path)
ref_ch = ops["ref_ch"]
ref_round = ops["ref_round"]
print(f"align_method={ops['align_method']}  ref_ch={ref_ch}  ref_round={ref_round}")
print(f"rounds_min_shift={ops['rounds_min_shift']}  rounds_max_shift={ops['rounds_max_shift']}")


def load_ref_stack(tile, prefix="barcode_round"):
    """Load a tile across every round of a modality -> (X, Y, channels, rounds)."""
    fam = prefix.split("_")[0]                 # "genes" or "barcode"
    nrounds = ops[prefix + "s"]                # ops["genes_rounds"] / ops["barcode_rounds"]
    projection = ops.get(f"{fam}_projection", "max")
    stack = load_sequencing_rounds(
        data_path, tile_coors=tuple(tile), prefix=prefix,
        suffix=projection, nrounds=nrounds,
    )
    return stack, nrounds, projection

In [ ]:
# STEP 1 — look at the reference channel of the tile across every round.
# This is the data the cross-round registration actually works with: a round
# that looks blank here is the one that will produce NaN shifts.
def show_rounds(tile, prefix="barcode_round"):
    stack, nrounds, proj = load_ref_stack(tile, prefix)
    ncol = 5
    nrow = int(np.ceil(nrounds / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3 * ncol, 3 * nrow))
    for r, ax in enumerate(axes.ravel()):
        if r < nrounds:
            img = stack[:, :, ref_ch, r]
            vmin, vmax = np.percentile(img, [1, 99.9])
            ax.imshow(img, cmap="gray", vmin=vmin, vmax=vmax)
            ax.set_title(f"{prefix.split('_')[0]} round {r}\nstd={img.std():.1f}", fontsize=9)
        ax.axis("off")
    fig.suptitle(f"ref_tile {tile}  ch{ref_ch}  ({prefix}, {proj})")
    plt.tight_layout()
    plt.show()


show_rounds([15, 3, 5], "barcode_round")

In [ ]:
# STEP 2 — run the actual reference-tile registration (same call as the pipeline's
# run_register_reference_tile, but without saving). Reports the per-round shifts of
# the reference channel and whether any transform is NaN (which would crash the run).
def run_ref_registration(tile, prefix="barcode_round", verbose=False):
    op = prefix.split("_")[0]
    stack, nrounds, proj = load_ref_stack(tile, prefix)
    print(f"=== {prefix}: ref_tile {tile}  ({nrounds} rounds, proj '{proj}') ===")
    angles, shifts, matrix = register_channels_and_rounds(
        stack,
        ref_ch=ref_ch,
        ref_round=ref_round,
        median_filter=ops["reg_median_filter"],
        min_shift=ops["rounds_min_shift"],
        max_shift=ops["rounds_max_shift"],
        align_method=ops["align_method"],
        use_masked_correlation=ops["use_masked_correlation"],
        reg_block_size=ops.get(f"{op}_reg_block_size", 256),
        reg_block_overlap=ops.get(f"{op}_reg_block_overlap", 0.5),
        correlation_threshold=ops.get(f"{op}_correlation_threshold", None),
        max_residual=ops.get(f"{op}_max_residual", 2),
        verbose=verbose,
    )
    has_nan = bool(np.isnan(shifts).any() or np.isnan(angles).any() or np.isnan(matrix).any())
    print(f"  per-round shifts of ch{ref_ch} (round {ref_round} is the reference) [dy, dx]:")
    for r in range(nrounds):
        s = shifts[ref_ch, r]
        flag = "  <-- NaN!" if np.isnan(s).any() else ""
        print(f"    round {r:2d}: [{s[0]:7.2f}, {s[1]:7.2f}]{flag}")
    verdict = "CONTAINS NaN -> would CRASH" if has_nan else "OK -> no NaN, registration succeeds"
    print(f"  >>> {verdict}\n")
    return dict(tile=tile, prefix=prefix, shifts=shifts, angles=angles,
                matrix=matrix, has_nan=has_nan)


_ = run_ref_registration([15, 3, 5], "barcode_round")

In [ ]:
# STEP 3 — compare both candidate ref_tiles on both modalities.
# (genes = 7 rounds, barcode = 14 rounds; barcode is the risky one.)
results = []
for tile in ([15, 3, 5], [15, 3, 2]):
    for prefix in ("genes_round", "barcode_round"):
        results.append(run_ref_registration(tile, prefix))

print("summary:")
for r in results:
    print(f"  {r['prefix']:14s} {r['tile']}  ->  {'NaN/CRASH' if r['has_nan'] else 'OK'}")